# Two-Stage Pill Identification - Training

**Stage 1:** Train YOLOv8n for pill detection  
**Stage 2:** Crop detected pills → Train ResNet-50 classifier  

Run cells top-to-bottom. Trained weights are saved back to Google Drive.

## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q ultralytics torch torchvision timm scikit-learn tqdm pandas opencv-python

In [ ]:
# Imports
import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from ultralytics import YOLO


In [ ]:
# Paths — update BASE_DIR if your Drive folder is in a different location
BASE_DIR    = '/content/drive/My Drive/dl proj/Pill_identification'
DATASET_DIR = f'{BASE_DIR}/prototype_dataset'
PROJECT_DIR = f'{BASE_DIR}/outputs_results'
CROPPED_DIR = f'{BASE_DIR}/cropped_pills'
CSV_PATH    = f'{BASE_DIR}/pill_detailed_contents.csv'
YAML_PATH   = f'{DATASET_DIR}/data.yaml'

print('Paths set:')
print(BASE_DIR, DATASET_DIR, PROJECT_DIR, CROPPED_DIR)


## 2. Data Preprocessing & Augmentation

Done via **Roboflow** before training:

**Preprocessing:** Auto-Orient, Resize (Stretch to 416×416)

**Augmentations:** 90° rotations, Crop (0–77% zoom), ±45° rotation, Shear ±15°, Hue/Saturation/Brightness/Exposure jitter, Blur up to 3px, Noise up to 5%

## 3. Stage 1 - YOLOv8 Training

Trains YOLOv8 nano (`yolov8n.pt`) on the pill detection dataset for 50 epochs.  
Best weights are saved automatically to `PROJECT_DIR/prototype1/weights/best.pt`.

In [ ]:
# Load pretrained YOLOv8n and fine-tune on the pill detection dataset
yolo_weights = 'yolov8n.pt'
model = YOLO(yolo_weights)

model.train(
    data=YAML_PATH,
    epochs=50,
    imgsz=416,
    batch=16,
    project=PROJECT_DIR,
    name='prototype1',
    exist_ok=True
)


## 4. Stage 1 - Crop Detected Pills

Uses the trained YOLO model to run inference on train/valid/test splits,  
crop each detected bounding box, and save the crops to `CROPPED_DIR`.  
These crops become the input dataset for the ResNet-50 classifier.

In [ ]:
# Load the best YOLO weights
yolo_best = YOLO(f"{PROJECT_DIR}/prototype1/weights/best.pt")

def crop_and_save(src_dir, out_dir):
    """Run YOLO on src_dir, crop each detection, save by class to out_dir."""
    os.makedirs(out_dir, exist_ok=True)
    results = yolo_best.predict(source=src_dir, save=False, conf=0.5)

    for i, r in enumerate(results):
        img = r.orig_img.copy()
        for j, (box, cls) in enumerate(zip(r.boxes.xyxy, r.boxes.cls)):
            x1, y1, x2, y2 = map(int, box)
            crop = img[y1:y2, x1:x2]

            cls_name   = yolo_best.names[int(cls)]
            cls_folder = os.path.join(out_dir, cls_name)
            os.makedirs(cls_folder, exist_ok=True)
            cv2.imwrite(f"{cls_folder}/img{i}_pill{j}.jpg", crop)

# Crop train / valid / test splits
crop_and_save(
    f'{DATASET_DIR}/train/images',
    f'{CROPPED_DIR}/train'
)
crop_and_save(
    f'{DATASET_DIR}/valid/images',
    f'{CROPPED_DIR}/valid'
)
crop_and_save(
    f'{DATASET_DIR}/test/images',
    f'{CROPPED_DIR}/test'
)

print('Cropping complete. Crops saved to:', CROPPED_DIR)


## 5. Stage 2 - DataLoaders for ResNet-50 Classifier

Loads the cropped pill images using `ImageFolder`.  
Applies standard ImageNet normalisation for ResNet-50.

In [ ]:
# Image transforms (ImageNet stats for ResNet-50)
TRANSFORMS = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'valid': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

train_dataset = datasets.ImageFolder(os.path.join(CROPPED_DIR, 'train'), TRANSFORMS['train'])
valid_dataset = datasets.ImageFolder(os.path.join(CROPPED_DIR, 'valid'), TRANSFORMS['valid'])

batch_size   = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

class_names = train_dataset.classes
print('Classes:', class_names)


## 6. Stage 2 - ResNet-50 Training

Loads pretrained ResNet-50 (ImageNet), freezes all base layers,  
and replaces the final FC layer with a custom head for our pill classes.  
Only the FC head is trained. Best model is saved to `BASE_DIR`.

In [ ]:
device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_classes = len(class_names)

# Load pretrained ResNet-50 and freeze base layers
model = models.resnet50(pretrained=True)
for p in model.parameters():
    p.requires_grad = False

# Replace the final FC layer for our number of classes
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, num_classes)
)
model = model.to(device)

# Loss and optimizer (only train the new FC head)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

epochs   = 10
best_acc = 0.0

for epoch in range(epochs):
    # ── Training phase ──
    model.train()
    running_loss = 0.0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # ── Validation phase ──
    model.eval()
    correct  = 0
    total    = 0
    val_loss = 0.0

    with torch.no_grad():
        for imgs, labels in valid_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out      = model(imgs)
            val_loss += criterion(out, labels).item()
            preds     = out.argmax(dim=1)
            correct  += (preds == labels).sum().item()
            total    += labels.size(0)

    val_acc = correct / total if total > 0 else 0
    print(f'Epoch {epoch+1}/{epochs} — '
          f'Train loss: {running_loss/len(train_loader):.4f} | '
          f'Val loss: {val_loss/len(valid_loader):.4f} | '
          f'Val acc: {val_acc:.4f}')

    # Save best model weights
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(
            model.state_dict(),
            os.path.join(BASE_DIR, 'pill_classifier_resnet50_best.pth')
        )
        print('  ✅ Saved best model')
